In [20]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import threading
import obspy
import os
import pickle
import numpy as np
from datetime import datetime
import time
from obspy.clients.seedlink.easyseedlink import create_client, EasySeedLinkClient

## realtime station information
http://ds.iris.edu/gmap/#network=_REALTIME&starttime=2021-03-01&datacenter=IRISDMC&networktype=permanent&planet=earth

http://ds.iris.edu/gmap/#network=_REALTIME&channel=HH*&starttime=2021-03-01&datacenter=IRISDMC&networktype=permanent&planet=earth

In [21]:
pi = 3.1415926
degree2km = pi*6371/180

## Location
# center = (-115.53, 32.98) #salton sea
# center = (-117.504, 35.705) #ridgecrest
center = (-155.32, 19.39) #havaii
horizontal_degree = 2.0
vertical_degree = 2.0

## Time range
#     starttime = obspy.UTCDateTime("2020-10-01T00") #salton sea
#     endtime = obspy.UTCDateTime("2020-10-03T00") ## not included
starttime = obspy.UTCDateTime("2021-01-01T00:00:00") 
endtime = obspy.UTCDateTime(datetime.utcnow())

## seismic stations
network_list = ["HV", "PT"]
channel_list = "HH*,HN*,BH*,EH*"

## data center
client = "IRIS"

####### save config ########
config = {}
config["center"] = center
config["xlim_degree"] = [center[0]-horizontal_degree/2, center[0]+horizontal_degree/2]
config["ylim_degree"] = [center[1]-vertical_degree/2, center[1]+vertical_degree/2]
config["degree2km"] = degree2km
config["starttime"] = starttime.datetime
config["endtime"] = endtime.datetime
config["networks"] = network_list
config["channels"] = channel_list
config["client"] = client

In [22]:
config

{'center': (-155.32, 19.39),
 'xlim_degree': [-156.32, -154.32],
 'ylim_degree': [18.39, 20.39],
 'degree2km': 111.19492474777779,
 'starttime': datetime.datetime(2021, 1, 1, 0, 0),
 'endtime': datetime.datetime(2024, 3, 11, 5, 32, 1, 613249),
 'networks': ['HV', 'PT'],
 'channels': 'HH*,HN*,BH*,EH*',
 'client': 'IRIS'}

In [23]:
stations_total = pd.read_csv("realtime-stations.txt", sep="|",  header=None, skiprows=3, names=["network", "station", "latitude", "longitude", "elevation(m)", "location", "starttime", "endtime"])
stations_total = stations_total[stations_total["network"].isin(network_list)]

plt.figure()
# plt.plot(stations_total["longitude"], stations_total["latitude"], '^')
# plt.axis("scaled")

stations_select = stations_total[(config["xlim_degree"][0] < stations_total["longitude"]) & (stations_total["longitude"] < config["xlim_degree"][1])\
& (config["ylim_degree"][0] < stations_total["latitude"]) & (stations_total["latitude"] < config["ylim_degree"][1])]

stations_select = stations_select.reset_index()
print(len(stations_select))
# plt.figure()
# plt.plot(stations_select["longitude"], stations_select["latitude"], '^')
# plt.axis("scaled")
# plt.show();

63


<Figure size 432x288 with 0 Axes>

In [24]:
import pickle, os
import obspy
from obspy.clients.fdsn import Client
from collections import defaultdict
import pandas as pd
#     import matplotlib
#     matplotlib.use("agg")
#     import matplotlib.pyplot as plt

####### Download stations ########
stations = Client("IRIS").get_stations(network = ",".join(config["networks"]),
                                       station = ",".join(stations_select["station"]),
                                       starttime=config["starttime"],
                                       endtime=config["endtime"],
                                       minlongitude=config["xlim_degree"][0],
                                       maxlongitude=config["xlim_degree"][1],
                                       minlatitude=config["ylim_degree"][0],
                                       maxlatitude=config["ylim_degree"][1],
                                       channel=config["channels"],
                                       level="response")#,
#                                            filename="stations.xml")

#     stations = obspy.read_inventory("stations.xml")
print("Number of stations: {}".format(sum([len(x) for x in stations])))
# stations.plot('local', outfile="stations.png")
# stations.plot('local')

####### Save stations ########
station_locs = defaultdict(dict)
station_resp = defaultdict(dict)
for network in stations:
    for station in network:
        for chn in station:
            sid = f"{network.code}.{station.code}.{chn.location_code}.{chn.code[:-1]}"
            station_resp[f"{network.code}.{station.code}.{chn.location_code}.{chn.code}"] =\
                chn.response.instrument_sensitivity.value
            if sid in station_locs:
                station_locs[sid]["component"] += f",{chn.code[-1]}"
                station_locs[sid]["response"] += f",{chn.response.instrument_sensitivity.value:.2f}"
            else:
                component = f"{chn.code[-1]}"
                response = f"{chn.response.instrument_sensitivity.value:.2f}"
                dtype = chn.response.instrument_sensitivity.input_units.lower()
                tmp_dict = {}
                tmp_dict["longitude"], tmp_dict["latitude"], tmp_dict["elevation(m)"] = chn.longitude, chn.latitude, chn.elevation
                tmp_dict["component"], tmp_dict["response"], tmp_dict["unit"] = component, response, dtype
                station_locs[sid] = tmp_dict

station_locs = pd.DataFrame.from_dict(station_locs, orient='index')
station_locs.to_csv("stations.csv",
                    sep="\t", float_format="%.3f",
                    index_label="station",
                    columns=["longitude", "latitude", "elevation(m)", "unit", "component", "response"])


#     ####### Plot stations ########
# plt.figure()
# plt.plot(station_locs["longitude"], station_locs["latitude"], "^", label="Stations")
# #     plt.plot(catalog["x(km)"], catalog["y(km)"], "k.", label="Earthquakes")
# plt.xlabel("X (km)")
# plt.ylabel("Y (km)")
# plt.axis("scaled")
# plt.legend()
# plt.title(f"Number of stations: {len(station_locs)}")
# #     plt.savefig(os.path.join(data_path, "stations_loc.png"))
# plt.show();

Number of stations: 62


In [25]:
timestamp = lambda x: x.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

In [18]:
class Client(EasySeedLinkClient):
    def __init__(self, server_url, producer, autoconnect=True):
        super().__init__(server_url, producer)
        self.producer = producer

    def on_data(self, trace):
        if time.time() % 10 < 0.5: ## print every 60s
            print(f'Received trace: {trace}')
        if trace.stats.sampling_rate != 100:
            trace = trace.interpolate(100, method="linear")
        if trace.stats.channel[1] == "N": ## acceleration
            trace = trace.integrate()
            trace = trace.filter("highpass", freq=1.0)
        value = {
            "timestamp": timestamp(trace.stats.starttime.datetime),
            "vec": (trace.data / station_resp[trace.id]).tolist(),
        }
        # self.producer.send('waveform_raw', key=trace.id, value=value)

In [19]:
client = Client('rtserve.iris.washington.edu:18000', producer=None)
for x in station_locs.index:
    x = x.split(".")
    client.select_stream(x[0], x[1], x[-1] + "?")
client.run()

KeyboardInterrupt: 